# Workshop 2: Retrieval Basics with RAG
## Theme Example: Scientific Articles (arXiv)

In this notebook, we work with scientific articles from the arXiv dataset.

Unlike plain text or PDFs, this dataset contains structured information such as:
- title  
- abstract  

We will focus on the abstracts, as they summarise the main ideas of each paper.

Our goal is to:
- load the dataset  
- extract useful text (title + abstract)  
- split into chunks  
- build a retrieval system  

## Working with Structured Data

Scientific datasets often contain multiple fields.

Instead of using the full dataset, we select only the most relevant parts:
- title → gives context  
- abstract → contains the main content  

We combine them into a single text for retrieval.

In [1]:
# Install required libraries (run once)
!pip install -q --upgrade langchain langchain-community langchain-text-splitters sentence-transformers faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [ ]:
!pip install -q kagglehub

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
import kagglehub
path = kagglehub.dataset_download("Cornell-University/arxiv")
print("Path:", path)

Using Colab cache for faster access to the 'arxiv' dataset.
Path: /kaggle/input/arxiv


In [ ]:
import json
data = []
file_path = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"
limit=5000 # limit as we cannot load the full dataset in Colab RAM
with open(file_path, "r") as f:
    for i, line in enumerate(f):
        if i >= limit:
            break
        try:
            data.append(json.loads(line))
        except:
            continue

print("Loaded papers:", len(data))
print(data[0])

Loaded papers: 5000
{'id': '0704.0001', 'submitter': 'Pavel Nadolsky', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'comments': '37 pages, 15 figures; published version', 'journal-ref': 'Phys.Rev.D76:013009,2007', 'doi': '10.1103/PhysRevD.76.013009', 'report-no': 'ANL-HEP-PR-07-12', 'categories': 'hep-ph', 'license': None, 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with da

## Preparing Text for Retrieval

Each paper contains multiple fields, but we focus on:
- title  
- abstract  

We combine them into a single text for each paper, which will be used in the retrieval system.

In [ ]:
documents = []

for item in data:
    text = item["title"] + " " + item["abstract"]
    documents.append(text)

print(documents[1][:500])

Sparsity-certifying Graph Decompositions   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a characterization of the family of $(k,\ell)$-sparse graphs and
algorithmic solutions to a family of problems concerning tree decompositions of
graphs. Special instances of sparse graphs appear in rigidity theory and have
received increased attention in recent years. In particular, our colored
pebbles generalize and strengthen the previous results of Lee and Streinu 


## Splitting the Papers into Chunks

The papers are longer than a simple sentence or paragraph, so we split them into smaller chunks.

This makes retrieval more effective and allows the system to compare smaller pieces of text with a query.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.create_documents(documents)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])

Number of chunks: 6726
Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies   A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. Th


In [ ]:
embedding_model = HuggingFaceEmbeddings()

db = FAISS.from_documents(chunks, embedding_model)

print("Vector store created successfully!")

/tmp/ipykernel_1085/3680806366.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings()
/tmp/ipykernel_1085/3680806366.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/s

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully!


## Retrieving Relevant Information

We can now search the paper collection using a query.

The system compares the query with the stored embeddings and returns the most relevant chunks.

In [ ]:
query = "neural networks for image classification"

results = db.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(r.page_content[:500])


Result 1:

Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Result 2:

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative 

Result 3:

The Parameter-Less Self-Organizing Map algorithm   The Parameter-Less Self-Organizing Map (PLSOM) is a new neural network
algorithm based on the Self-Organizing Map (SOM). It eliminates the need for a
learning rate and annealing schemes for learning rate and neighbourhood size.
We

In [ ]:
# Save FAISS index locally
db.save_local("faiss_Sci")

## Trying Different Queries

Try a few different queries and observe the retrieved results.

Think about:
- Are the results relevant?
- Which query works better?
- Do different queries retrieve different types of papers?

## Summary

In this notebook, we:
- loaded a scientific dataset  
- extracted title and abstract  
- split the text into chunks  
- created embeddings  
- stored them in a vector database  
- retrieved relevant information using queries  

This shows how retrieval can be applied to scientific literature.